# MATR Training on Google Colab
**Video-to-Video Moment Retrieval on SportsMoments dataset**

Run cells top-to-bottom. Features (~8GB) are cached in Google Drive so they survive session restarts.

**Recommended runtime:** A100 GPU (Colab Pro) or T4 (free tier, slower)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/MATR'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## 2. Clone Repo & Install Dependencies

In [ ]:
import os

REPO_DIR = '/content/MATR'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/vishnuDvardhan/MATR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install core dependencies (torch is pre-installed on Colab)
!pip install -q einops transformers==4.28.1 easydict tensorboard warmup-scheduler
!pip install -q git+https://github.com/facebookresearch/ImageBind

# ImageBind needs timm >= 0.9
!pip install -q 'timm>=0.9.0'

# CLIP for vision feature extraction
!pip install -q git+https://github.com/openai/CLIP.git

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Download SportsMoments Dataset

Downloads metadata + raw videos from HuggingFace. Videos are cached in Google Drive.

> **First run:** ~30–60 min to download ~15 GB of videos  
> **Subsequent runs:** symlinks from Drive, instant

In [ ]:
!pip install -q huggingface_hub datasets

import os
from pathlib import Path

DATA_DIR = f'{REPO_DIR}/data/sportsmr'
DRIVE_DATA = f'{DRIVE_ROOT}/data/sportsmr'

for sub in ['raw_target_videos', 'raw_query_videos', 'metadata', 'vid_clip', 'vid_clip_query', 'vid_imagebind', 'vid_imagebind_query']:
    os.makedirs(f'{DRIVE_DATA}/{sub}', exist_ok=True)

os.makedirs(DATA_DIR, exist_ok=True)

# Symlink each subfolder from Drive into the repo
for sub in ['raw_target_videos', 'raw_query_videos', 'metadata', 'vid_clip', 'vid_clip_query', 'vid_imagebind', 'vid_imagebind_query']:
    link = f'{DATA_DIR}/{sub}'
    target = f'{DRIVE_DATA}/{sub}'
    if not os.path.exists(link):
        os.symlink(target, link)
        print(f'Linked {link} -> {target}')
    else:
        print(f'Already exists: {link}')

In [ ]:
# Download metadata from HuggingFace
from huggingface_hub import hf_hub_download, list_repo_files
import shutil

HF_REPO = 'sportsmoments/SportsMoments'  # Update if different
META_DIR = f'{DATA_DIR}/metadata'

for split in ['train.jsonl', 'val.jsonl']:
    dest = f'{META_DIR}/{split}'
    if not os.path.exists(dest):
        print(f'Downloading {split}...')
        try:
            path = hf_hub_download(repo_id=HF_REPO, filename=f'metadata/{split}', repo_type='dataset', local_dir='/tmp/hf_dl')
            shutil.copy(path, dest)
            print(f'  Saved to {dest}')
        except Exception as e:
            print(f'  Error: {e}')
    else:
        print(f'  Already downloaded: {split}')

In [ ]:
# Download raw videos from HuggingFace
# This downloads in parallel using the huggingface_hub snapshot API
from huggingface_hub import snapshot_download
import shutil

def download_video_split(split_name, drive_dest):
    existing = len([f for f in os.listdir(drive_dest) if f.endswith('.mp4')])
    print(f'[{split_name}] {existing} videos already in Drive')
    if existing >= 4990:  # dataset has ~4998 per split
        print(f'  Skipping — already complete')
        return
    print(f'  Downloading {split_name} videos... (this may take 20-40 min)')
    try:
        dl_path = snapshot_download(
            repo_id=HF_REPO,
            repo_type='dataset',
            allow_patterns=[f'{split_name}/*.mp4'],
            local_dir='/tmp/hf_videos'
        )
        src = f'{dl_path}/{split_name}'
        if os.path.exists(src):
            for f in os.listdir(src):
                shutil.move(f'{src}/{f}', f'{drive_dest}/{f}')
            print(f'  Done: moved videos to Drive')
    except Exception as e:
        print(f'  Error: {e}')

download_video_split('raw_target_videos', f'{DRIVE_DATA}/raw_target_videos')
download_video_split('raw_query_videos', f'{DRIVE_DATA}/raw_query_videos')

## 4. Feature Extraction

Choose **one** of the two feature types below. ImageBind gives richer multimodal features (recommended).

Features are saved to Drive and reused across sessions — **only runs once**.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
FEAT_TYPE = 'imagebind'   # 'clip' (512-dim) or 'imagebind' (1024-dim)
# ───────────────────────────────────────────────────────────────────────

if FEAT_TYPE == 'clip':
    TARGET_FEAT_DIR = f'{DATA_DIR}/vid_clip'
    QUERY_FEAT_DIR  = f'{DATA_DIR}/vid_clip_query'
    V_FEAT_DIM      = 512
    T_FEAT_DIM      = 512
else:
    TARGET_FEAT_DIR = f'{DATA_DIR}/vid_imagebind'
    QUERY_FEAT_DIR  = f'{DATA_DIR}/vid_imagebind_query'
    V_FEAT_DIM      = 1024
    T_FEAT_DIM      = 1024

target_done = len([f for f in os.listdir(TARGET_FEAT_DIR) if f.endswith('.npz')])
query_done  = len([f for f in os.listdir(QUERY_FEAT_DIR)  if f.endswith('.npz')])
print(f'Feature type : {FEAT_TYPE}')
print(f'Target feats : {target_done} / ~4998')
print(f'Query feats  : {query_done}  / ~4998')

In [ ]:
# Extract CLIP features (run only if FEAT_TYPE == 'clip')
if FEAT_TYPE == 'clip':
    !PYTHONPATH={REPO_DIR} python {REPO_DIR}/extract_clip_features.py \
        --video_dir {DATA_DIR}/raw_target_videos \
        --out_dir {TARGET_FEAT_DIR} \
        --device cuda
    !PYTHONPATH={REPO_DIR} python {REPO_DIR}/extract_clip_features.py \
        --video_dir {DATA_DIR}/raw_query_videos \
        --out_dir {QUERY_FEAT_DIR} \
        --device cuda
else:
    print('Skipping CLIP extraction (FEAT_TYPE is imagebind)')

In [ ]:
# Extract ImageBind features (run only if FEAT_TYPE == 'imagebind')
if FEAT_TYPE == 'imagebind':
    # Cache weights in Drive
    IB_WEIGHTS_DRIVE = f'{DRIVE_ROOT}/imagebind_huge.pth'
    IB_WEIGHTS_LOCAL = f'{REPO_DIR}/.checkpoints/imagebind_huge.pth'
    os.makedirs(f'{REPO_DIR}/.checkpoints', exist_ok=True)
    if os.path.exists(IB_WEIGHTS_DRIVE) and not os.path.exists(IB_WEIGHTS_LOCAL):
        os.symlink(IB_WEIGHTS_DRIVE, IB_WEIGHTS_LOCAL)
        print('Linked ImageBind weights from Drive')

    !PYTHONPATH={REPO_DIR} python {REPO_DIR}/extract_imagebind_features.py \
        --video_dir {DATA_DIR}/raw_target_videos \
        --out_dir {TARGET_FEAT_DIR} \
        --device cuda

    !PYTHONPATH={REPO_DIR} python {REPO_DIR}/extract_imagebind_features.py \
        --video_dir {DATA_DIR}/raw_query_videos \
        --out_dir {QUERY_FEAT_DIR} \
        --device cuda

    # Save weights to Drive for future sessions
    if not os.path.exists(IB_WEIGHTS_DRIVE) and os.path.exists(IB_WEIGHTS_LOCAL):
        import shutil
        shutil.copy2(IB_WEIGHTS_LOCAL, IB_WEIGHTS_DRIVE)
        print('Cached ImageBind weights to Drive')
else:
    print('Skipping ImageBind extraction (FEAT_TYPE is clip)')

## 5. Training Config

In [ ]:
import subprocess, sys, os

# ── Training hyperparameters ────────────────────────────────────────────
EXP_ID          = 'colab-run-1'
N_EPOCH         = 30
BSZ             = 256        # reduce to 128 if OOM on T4
EVAL_BSZ        = 4
LR              = 1e-5
LR_DROP         = 24
LR_WARMUP       = 5
WD              = 1e-4
HIDDEN_DIM      = 1024
ENC_LAYERS      = 4
INPUT_DROPOUT   = 0.5
DROPOUT         = 0.0
DROPPATH        = 0.1
EVAL_EPOCH      = 2
NUM_WORKERS     = 4
GRAD_CLIP       = 0.1
# ───────────────────────────────────────────────────────────────────────

# Derived from FEAT_TYPE set above
V_FEAT_TYPES = FEAT_TYPE
T_FEAT_TYPE  = FEAT_TYPE

RESULTS_DIR = f'{DRIVE_ROOT}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Symlink results to Drive so checkpoints survive session
local_results = f'{REPO_DIR}/results'
if not os.path.exists(local_results):
    os.symlink(RESULTS_DIR, local_results)
    print(f'Linked results -> Drive')

print('Config ready.')
print(f'  exp_id      : {EXP_ID}')
print(f'  feat_type   : {FEAT_TYPE}  ({V_FEAT_DIM}-dim)')
print(f'  bsz         : {BSZ}')
print(f'  n_epoch     : {N_EPOCH}')
print(f'  results_dir : {RESULTS_DIR}')

## 6. Resume from Checkpoint (optional)

If resuming a previous run, set `RESUME_CKPT` to the checkpoint path. Otherwise leave as `None`.

In [ ]:
import glob

RESUME_CKPT = None  # e.g. f'{RESULTS_DIR}/mr-sportsmr/colab-run-1-.../model_best.ckpt'

# Auto-detect latest checkpoint for this exp_id
if RESUME_CKPT is None:
    pattern = f'{RESULTS_DIR}/mr-sportsmr/{EXP_ID}*/model_best.ckpt'
    matches = sorted(glob.glob(pattern))
    if matches:
        RESUME_CKPT = matches[-1]
        print(f'Auto-detected checkpoint: {RESUME_CKPT}')
    else:
        print('No checkpoint found — starting from scratch.')

## 7. Run Training

In [ ]:
import sys, os

# Build resume flag
resume_flag = f'--resume {RESUME_CKPT}' if RESUME_CKPT else ''

# Build arg string for shell command
args = (
    f"--dset_type mr --dset_name sportsmr --clip_length 2 "
    f"--exp_id {EXP_ID} --model_id MATR "
    f"--v_feat_types {V_FEAT_TYPES} --t_feat_type {T_FEAT_TYPE} "
    f"--ctx_mode video_tef "
    f"--train_path {DATA_DIR}/metadata/train.jsonl "
    f"--eval_path {DATA_DIR}/metadata/val.jsonl "
    f"--eval_split_name val "
    f"--v_feat_dirs {TARGET_FEAT_DIR} --v_feat_dim {V_FEAT_DIM} "
    f"--t_feat_dir {QUERY_FEAT_DIR} --t_feat_dim {T_FEAT_DIM} "
    f"--bsz {BSZ} --eval_bsz {EVAL_BSZ} --n_epoch {N_EPOCH} "
    f"--num_workers {NUM_WORKERS} --lr {LR} --lr_drop {LR_DROP} "
    f"--lr_warmup {LR_WARMUP} --wd {WD} "
    f"--hidden_dim {HIDDEN_DIM} --enc_layers {ENC_LAYERS} "
    f"--input_dropout {INPUT_DROPOUT} --dropout {DROPOUT} --droppath {DROPPATH} "
    f"--eval_epoch {EVAL_EPOCH} --use_cache -1 --easy_negative_only -1 "
    f"--main_metric MR-full-mAP-key --nms_thd 0.7 --max_before_nms 1000 "
    f"--b_loss_coef 1 --g_loss_coef 1 --eos_coef 0.1 --f_loss_coef 1 "
    f"--align_l 1 --s_loss_intra_coef 0 --s_loss_inter_coef 0 "
    f"--eval_mode add --round_multiple -1 --grad_clip {GRAD_CLIP} "
    f"--results_root {RESULTS_DIR} {resume_flag}"
)

print('Starting training...')
!cd {REPO_DIR} && PYTHONPATH={REPO_DIR} CUDA_VISIBLE_DEVICES=0 python main/train.py {args}

## 8. List Checkpoints

In [ ]:
import glob, os

ckpts = sorted(glob.glob(f'{RESULTS_DIR}/**/*.ckpt', recursive=True))
for c in ckpts:
    size_mb = os.path.getsize(c) / 1e6
    print(f'{size_mb:6.0f} MB  {c}')